# Notebook 1: BioBERT Entity Exrtraction & ICD Mapping

## Objective

- Fine-tune transformer models (BioBERT) on medical text
- Extract clinical entities to standardize medical codes (ICD-10)
- Map extracted entities from unstructured medical metrics
- Evaluate model performance with proper metrics

## Datasets Used

- MIMIC-III Clinical Database (primary training/testing data)
- Medical NER Dataset (evaluation gold standard) -- Using sythentic data for demo
- ICD-10 Codeset (Code mapping)

## Key Technologies

- `transformers` library (HuggingFace)
- BioBERT pre-traied model
- PyTorch
- pandas, numpy
- scikit-learn (metrics)


In [1]:
# Download datasets using kagglehub
import kagglehub

# MIMIC-III demo dataset
mimic_path = kagglehub.dataset_download("montassarba/mimic-iii-clinical-database-demo-1-4")
print("Path to MIMIC-III dataset files:", mimic_path)

# ICD-10 codeset
icd_path = kagglehub.dataset_download("mrhell/icd10cm-codeset-2023")
print("Path to ICD-10 dataset files:", icd_path)

D:\alexr\GitHub\clinical-nlp-claims-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████████████████████████████████████████| 10.6M/10.6M [00:00<00:00, 11.2MB/s]

Extracting files...


Path to MIMIC-III dataset files: C:\Users\alexr\.cache\kagglehub\datasets\montassarba\mimic-iii-clinical-database-demo-1-4\versions\1


100%|█████████████████████████████████████████████████████████████████████████████| 1.08M/1.08M [00:00<00:00, 7.41MB/s]

Extracting files...


Path to ICD-10 dataset files: C:\Users\alexr\.cache\kagglehub\datasets\mrhell\icd10cm-codeset-2023\versions\1


In [6]:
print(mimic_path)

C:\Users\alexr\.cache\kagglehub\datasets\montassarba\mimic-iii-clinical-database-demo-1-4\versions\1


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

# Load MIMIC-III notes (adjust path based on download)
notes_df = pd.read_csv(f'{mimic_path}/mimic-iii-clinical-database-demo-1.4/NOTEEVENTS.csv')

# Filter for discharge summaries
discharge_notes = notes_df[notes_df['CATEGORY'] == 'Discharge summary'].copy()

# Basic statistics
print(f"Total notes: {len(discharge_notes)}")
print(f"Average note length: {discharge_notes['TEXT'].str.len().mean():.0f} characters")

# Load ICD-10 codes (adjust path; assuming the file is named 'icd10.csv' in the dataset — rename if needed)
icd10_df = pd.read_csv(f'{icd_path}/icd10cm_codes_2023.txt', sep='\t', names=['code', 'description'])  # Adjust based on actual file format
print(f"Total ICD-10 codes: {len(icd10_df)}")

# Distribution of note lengths
plt.figure(figsize=(10, 6))
sns.histplot(discharge_notes['TEXT'].str.len(), bins=50)
plt.title('Distribution of Note Lengths')
plt.xlabel('Note Length (Characters)')
plt.show()

# Most common medical terms (word cloud)
all_text = ' '.join(discharge_notes['TEXT'])
words = re.findall(r'\w+', all_text.lower())
word_freq = Counter(words)
wc = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_freq)
plt.figure(figsize=(10, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Common Terms')
plt.show()

# Sample discharge summary with PHI redacted (simple redaction for demo)
sample_text = discharge_notes['TEXT'].iloc[0]
redacted_sample = re.sub(r'\[\*\*.*?\*\*\]', '[REDACTED]', sample_text)
print("Sample Redacted Discharge Summary:\n", redacted_sample[:500])  # First 500 chars

# ICD-10 code hierarchy visualization (simple count by category prefix)
icd10_df['category'] = icd10_df['code'].str[:3]
category_counts = icd10_df['category'].value_counts()
plt.figure(figsize=(12, 6))
category_counts[:20].plot(kind='bar')  # Top 20 categories
plt.title('ICD-10 Code Hierarchy (Top Categories)')
plt.xlabel('Category Prefix')
plt.ylabel('Count')
plt.show()

KeyError: 'CATEGORY'